# Run10 Alpha Training (Local)
Thin orchestration notebook for the canonical Run10 alpha-generation training path. 
Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/environment source files.


## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [1]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_BRANCH = None  # e.g. "feature/run10-alpha-overhaul"
INSTALL_REQUIREMENTS = False
RESET_OUTPUT_DIRS = True

TRAIN_REPO_CANDIDATES = [
    Path.cwd(),
    Path(r"C:\Users\Owner\tcn_tape_vectorized_version_clean"),
    Path("/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean"),
]
TRAIN_REPO_DIR = next((p for p in TRAIN_REPO_CANDIDATES if (p / ".git").exists()), None)

def run(cmd):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

if TRAIN_REPO_DIR is None:
    attempted = "\n".join(f" - {p}" for p in TRAIN_REPO_CANDIDATES)
    raise FileNotFoundError("Local repo not found. Tried:\n" + attempted)

if TRAIN_BRANCH:
    run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"])
    run(["git", "-C", str(TRAIN_REPO_DIR), "checkout", TRAIN_BRANCH])

if RESET_OUTPUT_DIRS:
    purge_paths = [
        TRAIN_REPO_DIR / "tcn_fusion_results",
        TRAIN_REPO_DIR / "tcn_results",
        TRAIN_REPO_DIR / "tcn_att_results",
        TRAIN_REPO_DIR / "output_log",
        TRAIN_REPO_DIR / "output_logs",
        TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
        TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
        TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",
        TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",
    ]
    for path in purge_paths:
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        elif path.exists():
            path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

if INSTALL_REQUIREMENTS:
    requirements_file = TRAIN_REPO_DIR / "requirements.txt"
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])

print("[OK] Local repo ready:", TRAIN_REPO_DIR)
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "HEAD"])
print("[OK] Requirements installed:", INSTALL_REQUIREMENTS)


[OK] Local repo ready: c:\Users\Owner\tcn_tape_vectorized_version_clean
+ git -C c:\Users\Owner\tcn_tape_vectorized_version_clean rev-parse --abbrev-ref HEAD
+ git -C c:\Users\Owner\tcn_tape_vectorized_version_clean rev-parse HEAD
[OK] Requirements installed: False


In [3]:
import tensorflow as tf
import subprocess

REQUIRE_GPU = False  # set True if you want hard fail without GPU

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if smi.returncode == 0:
        print("nvidia-smi:", [line.strip() for line in smi.stdout.splitlines() if line.strip()])
    else:
        print("nvidia-smi: not available")
except Exception:
    print("nvidia-smi: not available")

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)

if not gpus:
    msg = 'No GPU visible to TensorFlow; running on CPU (slower).'
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('[WARN]', msg)
else:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('TF build CUDA:', tf.test.is_built_with_cuda())


nvidia-smi: not available
TF GPUs: []
[WARN] No GPU visible to TensorFlow; running on CPU (slower).
Mixed precision policy: <DTypePolicy "float32">
TF build CUDA: False


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [4]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run10_alpha_config, assert_run10_alpha_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

RUN_ID = 'run10'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None


2026-03-12 14:17:30,649 - src.environment_tape_rl - INFO - [OK] TAPE Portfolio Environment loaded successfully
2026-03-12 14:17:30,650 - src.environment_tape_rl - INFO -    Key changes:
2026-03-12 14:17:30,651 - src.environment_tape_rl - INFO -    1. Reward = Portfolio Value (project baseline pattern)
2026-03-12 14:17:30,651 - src.environment_tape_rl - INFO -    2. Termination = Data exhausted only (no balance thresholds)
2026-03-12 14:17:30,652 - src.environment_tape_rl - INFO -    3. Action normalization = Softmax (numerically stable)
2026-03-12 14:17:30,652 - src.environment_tape_rl - INFO -    4. Portfolio math = Simple linear (no log-space)
2026-03-12 14:17:30,654 - src.environment_tape_rl - INFO -    5. No training wheels, no milestone rewards


In [7]:
import logging

QUIET_SRC_INFO_LOGS = True

if QUIET_SRC_INFO_LOGS:
    # Suppress noisy module INFO logs like: "... - src.environment_tape_rl - INFO ..."
    for logger_name in [
        "src",
        "src.environment_tape_rl",
        "src.notebook_helpers.tcn_phase1",
        "src.agents.ppo_agent_tf",
    ]:
        logging.getLogger(logger_name).setLevel(logging.WARNING)
    print("[OK] Suppressed src INFO logs (level=WARNING).")
else:
    print("[INFO] Src INFO logs left enabled.")


[OK] Suppressed src INFO logs (level=WARNING).


## 3) Build Canonical Run10 Config and Dataset
Create the source-backed Run10 config, assert no drift, and prepare the dataset once.


In [5]:
train_config = build_run10_alpha_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run10_alpha_config(train_config)

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']

print('[Run10] Canonical config ready')
print('  analysis_start_date =', train_config['ANALYSIS_START_DATE'])
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  regime_conditioning =', ap['regime_conditioning_enabled'])
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  cvar_advantage_weight =', ppo['cvar_advantage_weight'])
print('  lagrangian =', {
    'threshold': ppo['lagrangian_cvar_threshold'],
    'lr': ppo['lagrangian_cvar_lr'],
    'lambda_max': ppo['lagrangian_cvar_lambda_max'],
    'penalty_scale': ppo['lagrangian_cvar_penalty_scale'],
})
print('  drawdown =', {
    'target': env['drawdown_constraint']['target'],
    'tolerance': env['drawdown_constraint']['tolerance'],
    'penalty_coef': env['drawdown_constraint']['penalty_coef'],
    'lambda_carry_decay': env['drawdown_constraint']['lambda_carry_decay'],
})
print('  dispersion =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'dispersion_target_std': ppo['alpha_dispersion_target_std'],
})
print('  deterministic_validation =', tp['deterministic_validation_checkpointing_enabled'])
print('  training_early_stop =', tp.get('training_early_stop_enabled', False))

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
if actuarial_cols:
    raise RuntimeError(f'Actuarial columns should be absent for Run10: {actuarial_cols}')

alpha_ret_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('AlphaRet_')]
expected_alpha_cols = {'AlphaRet_1d', 'AlphaRet_5d', 'AlphaRet_20d', 'AlphaRet_5d_Z', 'AlphaRet_20d_Z'}
missing_alpha_cols = sorted(list(expected_alpha_cols - set(alpha_ret_cols)))
if missing_alpha_cols:
    raise RuntimeError(f'Missing expected alpha-return columns: {missing_alpha_cols}')

fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Actuarial feature check passed: none present (disabled)')
print('[OK] Alpha-return feature check passed:', sorted(alpha_ret_cols)[:10])
print('[OK] Fundamental feature check passed: none present')


2026-03-12 14:17:43,154 - src.data_utils - INFO - DataProcessor initialized for 10 assets: ['MSFT', 'NVDA', 'AMZN', 'JPM', 'CAT', 'XOM', 'JNJ', 'PG', 'GLD', 'NEE']
2026-03-12 14:17:43,155 - src.data_utils - INFO - Downloading comprehensive market data...


[Run10] Canonical config ready
  split_date = 2019-12-31
  architecture = TCN_FUSION
  regime_conditioning = True
  distributional_critic = True
  cvar_advantage_weight = 0.1
  lagrangian = {'threshold': -0.025, 'lr': 0.004, 'lambda_max': 5.0, 'penalty_scale': 5.0}
  drawdown = {'target': 0.15, 'tolerance': -0.01, 'penalty_coef': 2.5, 'lambda_carry_decay': 0.4}
  dispersion = {'hhi_coef': 0.01, 'dispersion_coef': 0.05, 'dispersion_target_std': 0.07}
  deterministic_validation = True
📊 Loading raw market data...


2026-03-12 14:17:54,670 - src.data_utils - INFO - Filtered data from 2003-09-02
2026-03-12 14:17:54,671 - src.data_utils - INFO - Filtered data to 2025-09-01
2026-03-12 14:17:54,682 - src.data_utils - INFO - Filtered to configured assets: ['MSFT', 'NVDA', 'AMZN', 'JPM', 'CAT', 'XOM', 'JNJ', 'PG', 'GLD', 'NEE']
2026-03-12 14:17:55,224 - src.data_utils - INFO - 💾 Cached OHLCV data to c:\Users\Owner\tcn_tape_vectorized_version_clean\data\daily_ohlcv_assets.csv
2026-03-12 14:17:55,225 - src.data_utils - INFO - [OK] Successfully processed market data. Shape: (55043, 7)
2026-03-12 14:17:55,228 - src.data_utils - INFO - 📅 Date range: 2003-09-02 00:00:00 to 2025-08-29 00:00:00
2026-03-12 14:17:55,235 - src.data_utils - INFO - 📊 Tickers: ['MSFT', 'NVDA', 'AMZN', 'JPM', 'CAT', 'XOM', 'JNJ', 'PG', 'GLD', 'NEE']
2026-03-12 14:17:55,260 - src.data_utils - INFO - Calculating log returns for periods: [1, 5, 10, 21]
2026-03-12 14:17:55,296 - src.data_utils - INFO - Period 1d: 55033/55043 valid returns

   [OK] Raw data shape: (55043, 7)
   [OK] Date range: 2003-09-02 00:00:00 => 2025-08-29 00:00:00

[TOOL] Computing multi-horizon log returns: [1, 5, 10, 21]
   [OK] Shape after returns: (54833, 11)

📈 Calculating 21-day rolling statistics


2026-03-12 14:18:00,876 - src.data_utils - INFO - Calculating 6 technical indicators for 10 assets


...

   [OK] Dates after filter: 5513 trading days
   [OK] Date range: 2003-10-02 00:00:00 to 2025-08-29 00:00:00
[SPLIT]  TIME-BASED TRAIN/TEST SPLIT (Fixed date: 2019-12-31)
   Train: 2003-10-02 => 2019-12-31 (4090 days, 16.2 years, 40593 rows)
   Test:  2020-01-02 => 2025-08-29 (1423 days, 5.6 years, 14230 rows)

[TOOL] NORMALISING FEATURES (standard scaler)


2026-03-12 14:18:58,275 - src.data_utils - INFO - Training data for scaler fitting: 2003-10-02 00:00:00 to 2019-12-31 00:00:00
2026-03-12 14:18:58,279 - src.data_utils - INFO - Training samples: 40593, Total samples: 54823
2026-03-12 14:18:58,283 - src.data_utils - INFO - Training unique dates: 4090, Total unique dates: 5513
2026-03-12 14:18:58,297 - src.data_utils - INFO - [OK] LogReturn_1d: applied robust+winsor normalization (p0.5=-0.065357, p99.5=0.067619)
2026-03-12 14:18:58,305 - src.data_utils - INFO - [OK] LogReturn_5d: applied robust+winsor normalization (p0.5=-0.144701, p99.5=0.139458)
2026-03-12 14:18:58,318 - src.data_utils - INFO - [OK] LogReturn_10d: applied robust+winsor normalization (p0.5=-0.202420, p99.5=0.189664)
2026-03-12 14:18:58,332 - src.data_utils - INFO - [OK] LogReturn_21d: applied robust+winsor normalization (p0.5=-0.286706, p99.5=0.273867)
2026-03-12 14:18:58,344 - src.data_utils - INFO - [OK] RollingVolatility_21d: applied robust+winsor normalization (p0.5


💾 Saving NORMALISED master dataframe to 'c:\Users\Owner\tcn_tape_vectorized_version_clean\data\master_features_NORMALIZED.csv'


2026-03-12 14:19:33,294 - src.data_utils - INFO - Scalers saved to c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_scalers.joblib



💾 Saved preparation artifacts:
   raw OHLCV: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_raw_ohlcv.csv
   full engineered: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_feature_engineered_full.csv
   analysis-window engineered: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_feature_engineered_analysis_window.csv
   normalized master: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_feature_engineered_normalized.csv
   train normalized: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_train_normalized.csv
   test normalized: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_test_normalized.csv
   scalers: c:\Users\Owner\tcn_tape_vectorized_version_clean\data_exports\phase1_prep_20260312_141907_scalers.joblib
   audit report: c:\Users\O

## 4) Run Training
Launch the canonical Run10 training path.


In [8]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


[START] Starting training
Architecture: TCN_FUSION
max_total_timesteps: 500000
num_parallel_envs: 4

EXPERIMENT 6: TCN_FUSION Enhanced + TAPE Three-Component
Architecture: TCN + Fusion
Results root: C:\Users\Owner\tcn_tape_vectorized_version_clean\tcn_fusion_results
Working dir: C:\Users\Owner\tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe <= 0.00 or MDD >= 25.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   [CYCLE] Profile Manager: disabled (static profile only)
[RAND] Experiment Seed: 6042 (Base: 42, Offset: 6000)
[OK] Features: Enhanced (includes 2 covariance eigenvalues)
   Eigenvalues: ['Covariance_Eigenvalue_0', 'Covariance_Eigenvalue_1']
   Train shape: (40593, 66)
   Test shape: (14230, 66)
   🧮 Actuarial colum

KeyboardInterrupt: 

## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [ ]:
TRAIN_RESULTS_ROOT = TRAIN_REPO_DIR / 'tcn_fusion_results'
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


## 6) Export Artifacts (Optional)
Zip the latest results and optionally copy them to Google Drive.


In [ ]:
import shutil
import subprocess

EXPORT_RESULTS_ZIP = False
COPY_TO_DIR = None  # Example: Path("/mnt/c/Users/Owner/Downloads")
EXPORT_PATH = TRAIN_REPO_DIR / f"tcn_tape_vectorized_{RUN_ID}.zip"

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_REPO_DIR / "tcn_fusion_results",
        TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
        TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
        TRAIN_REPO_DIR / "data_exports",
        TRAIN_REPO_DIR / "output_log",
        TRAIN_REPO_DIR / "output_logs",
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            [
                "bash",
                "-lc",
                "cd \"{}\" && zip -qr \"{}\" {}".format(
                    TRAIN_REPO_DIR,
                    EXPORT_PATH,
                    " ".join(f'\"{item}\"' for item in relative_items),
                ),
            ],
            check=True,
        )
        print("[OK] Created:", EXPORT_PATH)

        if COPY_TO_DIR is not None:
            copy_dir = Path(COPY_TO_DIR)
            copy_dir.mkdir(parents=True, exist_ok=True)
            target = copy_dir / EXPORT_PATH.name
            shutil.copy2(EXPORT_PATH, target)
            print("[OK] Copied to:", target)
    else:
        print("[WARN] Nothing to export.")
else:
    print("[SKIP] EXPORT_RESULTS_ZIP=False")
